# 🔵 Clustering — Concepts Notebook
### K-Means, Hierarchical aur DBSCAN — Sample Data se Samjho

---

**Topics Covered:**
1. 🧩 Introduction to Clustering — Unsupervised Learning
2. 📍 K-Means Clustering — Theory aur Manual Steps
3. 📐 Elbow Method aur Silhouette Score
4. 🌳 Hierarchical Clustering aur Dendrogram
5. 🔵 DBSCAN — Density-based Clustering
6. ⚖️ Comparison — Kab Kya Use Karein
7. 🏆 Capstone Exercise

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('✅ Ready!')

---
## 🧩 Section 1: Introduction to Clustering

**Naye Keywords:**
- **Unsupervised Learning** — labels ke bina patterns dhundna
- **Clustering** — similar data points ko groups (clusters) mein organize karna
- **Inertia** — cluster ke andar points aur centroid ke beech total distance (kam = better)
- **Centroid** — cluster ka center point
- **Applications:** Customer segmentation, image compression, anomaly detection

In [ ]:
from sklearn.datasets import make_blobs

# Sample data: 150 points, 3 natural groups
np.random.seed(42)
X, y_true = make_blobs(n_samples=150, centers=3, cluster_std=0.8, random_state=42)

# Also create a customer dataset for context
customers = pd.DataFrame({
    'Customer': [f'C{i}' for i in range(1, 11)],
    'Annual_Spend_K': [2, 3, 2.5, 8, 9, 8.5, 15, 16, 14, 2.8],
    'Visit_Freq': [1, 2, 1, 5, 6, 5, 10, 11, 9, 2]
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X[:,0], X[:,1], c='gray', alpha=0.6)
axes[0].set_title('Raw Data — Groups not labeled yet')

axes[1].scatter(customers['Annual_Spend_K'], customers['Visit_Freq'], c='steelblue', s=100)
for _, row in customers.iterrows():
    axes[1].annotate(row['Customer'], (row['Annual_Spend_K'], row['Visit_Freq']), fontsize=8)
axes[1].set_xlabel('Annual Spend (K)')
axes[1].set_ylabel('Visit Frequency')
axes[1].set_title('Customer Data — Natural Groups?')

plt.tight_layout()
plt.show()

---
## 📍 Section 2: K-Means Clustering — Manual Steps

**Naye Keywords:**
- **K** — kitne clusters banana hai (hum decide karte hain)
- **Initialization** — pehle random centroids choose karna
- **Assignment Step** — har point ko nearest centroid ka cluster assign karna
- **Update Step** — cluster ke sab points ka mean nikalna = new centroid
- **Convergence** — jab centroids move karna band kar dein
- **KMeans++** — smart initialization (random se better)

In [ ]:
# --- Manual K-Means on small 2D data ---
points = np.array([
    [1, 1], [1.5, 2], [3, 4], [5, 7],
    [3.5, 5], [4.5, 5], [3.5, 4.5]
], dtype=float)

# Step 1: Initialize centroids (randomly pick 2 points)
np.random.seed(0)
centroids = points[np.random.choice(len(points), size=2, replace=False)].copy()
print("Initial centroids:")
print(f"  C1 = {centroids[0]}")
print(f"  C2 = {centroids[1]}")

for iteration in range(5):
    # Step 2: Assign each point to nearest centroid
    distances = np.array([[np.linalg.norm(p - c) for c in centroids] for p in points])
    assignments = np.argmin(distances, axis=1)

    # Step 3: Update centroids
    new_centroids = np.array([points[assignments == k].mean(axis=0) for k in range(2)])

    shift = np.sum(np.abs(new_centroids - centroids))
    centroids = new_centroids

    print(f"\nIteration {iteration+1}: Assignments = {assignments}, Centroid shift = {shift:.4f}")
    if shift < 0.001:
        print("✅ Converged!")
        break

print(f"\nFinal Centroids: C1={centroids[0].round(2)}, C2={centroids[1].round(2)}")

In [ ]:
# --- Visualize K-Means iterations ---
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)
centers = kmeans.cluster_centers_

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#e74c3c', '#3498db', '#2ecc71']

axes[0].scatter(X[:,0], X[:,1], c='gray', alpha=0.5)
axes[0].set_title('Before Clustering')

for k in range(3):
    mask = labels == k
    axes[1].scatter(X[mask,0], X[mask,1], c=colors[k], alpha=0.7, label=f'Cluster {k}')
axes[1].scatter(centers[:,0], centers[:,1], c='black', s=200, marker='*', zorder=5, label='Centroids')
axes[1].set_title(f'K-Means Result (K=3, Inertia={kmeans.inertia_:.1f})')
axes[1].legend()

plt.suptitle('K-Means Clustering', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📐 Section 3: Elbow Method aur Silhouette Score

**Naye Keywords:**
- **Elbow Method** — inertia vs K plot mein 'elbow' point = optimal K
- **Inertia** — total within-cluster sum of squares (WCSS)
- **Silhouette Score** — cluster quality measure (-1 se 1, higher = better)
  - s = (b - a) / max(a, b)
  - a = average distance to same cluster points
  - b = average distance to nearest other cluster

In [ ]:
from sklearn.metrics import silhouette_score

# Elbow Method
inertias = []
silhouettes = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Elbow plot
axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2)
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='Optimal K=3')
axes[0].set_xlabel('Number of Clusters K')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method')
axes[0].legend()

# Silhouette plot
axes[1].plot(list(k_range), silhouettes, 'go-', linewidth=2)
axes[1].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='Optimal K=3')
axes[1].set_xlabel('Number of Clusters K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
axes[1].legend()

plt.suptitle('Finding Optimal K', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

best_k = list(k_range)[silhouettes.index(max(silhouettes))]
print(f"Best K by Silhouette: {best_k} (score = {max(silhouettes):.4f})")

---
## 🌳 Section 4: Hierarchical Clustering

**Naye Keywords:**
- **Hierarchical Clustering** — tree structure banata hai (dendrogram)
- **Agglomerative** — bottom-up: pehle sab alag, phir merge karte jao
- **Divisive** — top-down: pehle sab ek, phir split karte jao
- **Dendrogram** — clustering process ka tree visualization
- **Linkage** — do clusters ke beech distance kaise measure karein
  - **single** — nearest points
  - **complete** — farthest points
  - **average** — average distance
  - **ward** — variance minimize karna (most common)

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering

# Small dataset for dendrogram clarity
X_small, _ = make_blobs(n_samples=20, centers=3, cluster_std=0.6, random_state=42)

# Linkage matrix
Z = linkage(X_small, method='ward')

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
dendrogram(Z, leaf_rotation=90, leaf_font_size=8)
plt.axhline(y=4, color='red', linestyle='--', label='Cut = 3 clusters')
plt.title('Dendrogram (Ward Linkage)')
plt.xlabel('Sample Index')
plt.ylabel('Distance')
plt.legend()

# Agglomerative result
plt.subplot(1, 2, 2)
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
agg_labels = agg.fit_predict(X_small)
colors = ['#e74c3c', '#3498db', '#2ecc71']
for k in range(3):
    mask = agg_labels == k
    plt.scatter(X_small[mask, 0], X_small[mask, 1], c=colors[k], s=80, label=f'Cluster {k}')
plt.title('Hierarchical Clustering Result (k=3)')
plt.legend()

plt.tight_layout()
plt.show()

---
## 🔵 Section 5: DBSCAN

**DBSCAN = Density-Based Spatial Clustering of Applications with Noise**

**Naye Keywords:**
- **eps** — ek point ke aas-paas kitni radius consider karni hai
- **min_samples** — ek point ko core point kehne ke liye minimum neighbors
- **Core Point** — eps radius mein min_samples neighbors hain
- **Border Point** — core point ke neighborhood mein hai lekin khud core nahi
- **Noise Point** — kisi cluster ka part nahi (-1 label)
- **Advantage** — arbitrary shapes ke clusters dhundh sakta hai, outliers detect karta hai
- **K-Means se Farq** — K specify nahi karna, irregular shapes handle karta hai

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons, make_circles

# DBSCAN on non-linear shapes
X_moons, _ = make_moons(n_samples=200, noise=0.1, random_state=42)
X_circles, _ = make_circles(n_samples=200, noise=0.05, factor=0.5, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

datasets = [
    (X_moons, 'Two Moons'),
    (X_circles, 'Concentric Circles')
]

for row, (data, name) in enumerate(datasets):
    # K-Means (fails on non-linear)
    km_l = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(data)
    # DBSCAN (works!)
    db_l = DBSCAN(eps=0.2, min_samples=5).fit_predict(data)

    axes[row, 0].scatter(data[:,0], data[:,1], c='gray', alpha=0.5)
    axes[row, 0].set_title(f'{name} — Raw')

    axes[row, 1].scatter(data[:,0], data[:,1], c=km_l, cmap='RdBu', alpha=0.7)
    axes[row, 1].set_title('K-Means (fails here)')

    cmap_colors = {-1: 'gray', 0: '#e74c3c', 1: '#3498db', 2: '#2ecc71'}
    point_colors = [cmap_colors.get(l, '#9b59b6') for l in db_l]
    axes[row, 2].scatter(data[:,0], data[:,1], c=point_colors, alpha=0.7)
    n_noise = sum(db_l == -1)
    axes[row, 2].set_title(f'DBSCAN (noise={n_noise})')

plt.suptitle('K-Means vs DBSCAN on Non-Linear Shapes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- DBSCAN Hyperparameter Effect ---
eps_values = [0.1, 0.2, 0.5]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, eps in zip(axes, eps_values):
    db = DBSCAN(eps=eps, min_samples=5)
    db_labels = db.fit_predict(X_moons)
    n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
    n_noise = sum(db_labels == -1)
    
    unique_labels = set(db_labels)
    palette = plt.cm.Set1(np.linspace(0, 1, max(len(unique_labels), 1)))
    for i, label in enumerate(sorted(unique_labels)):
        mask = db_labels == label
        color = 'gray' if label == -1 else palette[i]
        ax.scatter(X_moons[mask, 0], X_moons[mask, 1], c=[color]*sum(mask), s=20)
    ax.set_title(f'eps={eps}\nClusters={n_clusters}, Noise={n_noise}')

plt.suptitle('DBSCAN: Effect of eps Parameter', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Insight: eps too small = too many noise points; eps too large = all merged")

---
## ⚖️ Section 6: Algorithm Comparison

| Feature | K-Means | Hierarchical | DBSCAN |
|---|---|---|---|
| K specify karna | ✅ Required | ❌ Optional (cut) | ❌ Not needed |
| Shape | Spherical only | Flexible | Any shape |
| Outliers | Sensitive | Moderate | Handles well (-1) |
| Scalability | Fast (large data) | Slow (O(n²)) | Medium |
| Deterministic | No (random init) | Yes | Yes |
| Interpretability | Easy | Dendrogram | Medium |

**Rule of thumb:**
- Round-shaped clusters, known K → K-Means
- Hierarchy important → Hierarchical  
- Arbitrary shapes, outliers → DBSCAN

---
## 🏆 Capstone + Solutions

In [ ]:
# Mall Customer Data
np.random.seed(42)
mall = pd.DataFrame({
    'Age': np.concatenate([np.random.randint(18,30,30), np.random.randint(30,50,30), np.random.randint(50,70,30)]),
    'Annual_Income_K': np.concatenate([np.random.randint(15,40,30), np.random.randint(40,80,30), np.random.randint(80,120,30)]),
    'Spending_Score': np.concatenate([np.random.randint(60,90,30), np.random.randint(40,70,30), np.random.randint(10,40,30)])
})

print("Mall Customer Data:")
print(mall.head())
print("Shape:", mall.shape)

# TODO A: StandardScaler lagao
# TODO B: K-Means (k=3) lagao, inertia print karo
# TODO C: Silhouette score calculate karo
# TODO D: Elbow method plot banao (k=2 se 8)
# TODO E: DBSCAN lagao, noise points count karo

print("\nTODO A-E complete karo!")

In [ ]:
# ===== SOLUTIONS =====
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# SOLUTION A: Scaling
scaler = StandardScaler()
mall_scaled = scaler.fit_transform(mall)

# SOLUTION B: K-Means
km_mall = KMeans(n_clusters=3, random_state=42, n_init=10)
mall['Cluster'] = km_mall.fit_predict(mall_scaled)
print("K-Means Inertia:", round(km_mall.inertia_, 2))

# SOLUTION C: Silhouette
sil = silhouette_score(mall_scaled, mall['Cluster'])
print("Silhouette Score:", round(sil, 4))

# SOLUTION D: Elbow
inertias = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(mall_scaled).inertia_ for k in range(2, 9)]
plt.figure(figsize=(6, 3))
plt.plot(range(2, 9), inertias, 'bo-')
plt.xlabel('K')
plt.ylabel('Inertia')
plt.title('Elbow Method — Mall Customers')
plt.tight_layout()
plt.show()

# SOLUTION E: DBSCAN
db_mall = DBSCAN(eps=0.5, min_samples=5).fit_predict(mall_scaled)
print("DBSCAN Noise points:", sum(db_mall == -1))
print("DBSCAN Clusters:", len(set(db_mall)) - (1 if -1 in db_mall else 0))

---
## 🎓 Summary

| Algorithm | Key Idea | Best For |
|---|---|---|
| K-Means | Centroid-based, minimize inertia | Customer segmentation, image compression |
| Hierarchical | Tree-based merging/splitting | Gene expression, document clustering |
| DBSCAN | Density-based, finds arbitrary shapes | Geospatial data, anomaly detection |